**Phase 1:Exploring Data Foundation & Viewing patterns in the dataset**

In [36]:
import pandas as pd
import matplotlib as mp
current_week = pd.read_csv("current_week.csv")
current_week=pd.DataFrame(current_week)
print("=== Current Week Data ===")
current_week.head(5)

initial_stock = pd.read_csv("initial_stock.csv")
initial_stock=pd.DataFrame(initial_stock)
print("=== Initial Stock Data ===")
initial_stock.head(5)

last_week = pd.read_csv("last_week.csv")

print("=== Last Week Data ===")
last_week.head(5)

print("Total rows in Current Week Data:", len(current_week))


=== Current Week Data ===
=== Initial Stock Data ===
=== Last Week Data ===
Total rows in Current Week Data: 193


In [37]:
## counting numbe rof rows in each dataset
print("Number of rows in Current Week Data:", len(current_week))
print("Number of rows in Last Week Data:", len(last_week))
print("Number of rows in Initial Stock Data:", len(initial_stock))


Number of rows in Current Week Data: 193
Number of rows in Last Week Data: 159
Number of rows in Initial Stock Data: 6


In [38]:
##Finding the number of customers 
num_customers = current_week['customer_id'].nunique()
print("Number of unique customers in Current Week Data:", num_customers)

Number of unique customers in Current Week Data: 53


In [39]:
##Date range of the data for each week
print("Date range for Current Week Data:", current_week['order_date'].min(), "to", current_week['order_date'].max())
print("Date range for Last Week Data:", last_week['order_date'].min(), "to", last_week['order_date'].max())


Date range for Current Week Data: 2026-04-20 to 2026-04-26
Date range for Last Week Data: 2026-04-20 to 2026-04-26


In [40]:
##Which product appears most from list during each week and which appeared least number of times
most_frequent_product_last_week= last_week['product_name'].value_counts().idxmax()
count_frequent_last=(last_week["product_name"]==most_frequent_product_last_week).sum()
least_frequent_product_last_week = last_week['product_name'].value_counts().idxmin()
count_least_frequent_last=(last_week["product_name"]==least_frequent_product_last_week).sum()
print("most frequent product in Last Week Data:", most_frequent_product_last_week,"which was purchased",count_frequent_last,"times")
print("least frequent product in Last Week Data:", least_frequent_product_last_week,"which was purchased",count_least_frequent_last,"times")

most_frequent_product_current_week = current_week['product_name'].value_counts().idxmax()
count_frequent_current=(current_week["product_name"]==most_frequent_product_current_week).sum()
least_frequent_product_current_week = current_week['product_name'].value_counts().idxmin()
count_least_frequent_current=(current_week["product_name"]==least_frequent_product_current_week).sum()
print("most frequent product in Current Week Data:", most_frequent_product_current_week,"which was purchased",count_frequent_current,"times")
print("least frequent product in Current Week Data:", least_frequent_product_current_week,"which was purchased",count_least_frequent_current,"times")

most frequent product in Last Week Data: Phone Case which was purchased 48 times
least frequent product in Last Week Data: Portable Power Bank which was purchased 13 times
most frequent product in Current Week Data: Phone Case which was purchased 55 times
least frequent product in Current Week Data: Laptop Sleeve which was purchased 15 times


In [41]:
##Total revenue generated in each week
total_revenue_last_week = (last_week['quantity'] * last_week['unit_price_pkr']).sum()
total_revenue_current_week = (current_week['quantity'] * current_week['unit_price_pkr']).sum()
print("Total revenue generated in Last Week Data:", total_revenue_last_week)
print("Total revenue generated in Current Week Data:", total_revenue_current_week)

Total revenue generated in Last Week Data: 300270
Total revenue generated in Current Week Data: 386285


In [42]:
print(current_week['is_return'])

0      False
1      False
2      False
3      False
4      False
       ...  
188    False
189    False
190    False
191    False
192    False
Name: is_return, Length: 193, dtype: bool


In [43]:
# Use .sum() to count True values and dynamic length for total count
return_rate_last_week = (last_week['is_return'].sum() / len(last_week)) * 100
return_rate_current_week = (current_week['is_return'].sum() / len(current_week)) * 100

print("Return rate for Last Week Data: {:.2f}%".format(return_rate_last_week))
print("Return rate for Current Week Data: {:.2f}%".format(return_rate_current_week))


Return rate for Last Week Data: 18.87%
Return rate for Current Week Data: 12.95%


**What can be seen from the Data**

Weekly Sales Analysis Inferences
Total Sales Volume: The number of sales in the current week is lower than the sales recorded in the previous week.

Product Performance Gap: Current week performance is significantly down, as indicated by the fact that the most frequent product's sales this week are lower than the least frequent product's sales from the prior period.

Return Rate: There has been an upward trend in product returns; the return rate for this week is higher than it was last week.

**Phase 2:To go in depth of our obeservations related to Product sales**


In [44]:
##Revenue difference for each product between the two weeks 
dif_byproduct_eachweek = initial_stock[['product_name']].drop_duplicates()


revenue_current = current_week.groupby('product_name').apply(
    lambda x: (x['quantity'] * x['unit_price_pkr']).sum()
).reset_index(name='Revenue in current week')


revenue_last = last_week.groupby('product_name').apply(
    lambda x: (x['quantity'] * x['unit_price_pkr']).sum()
).reset_index(name='Revenue in previous week')


dif_byproduct_eachweek =dif_byproduct_eachweek .merge(revenue_current, on='product_name', how='left')
dif_byproduct_eachweek =dif_byproduct_eachweek .merge(revenue_last, on='product_name', how='left').fillna(0)

# 4. Difference in the revenue
dif_byproduct_eachweek ['difference'] = dif_byproduct_eachweek ['Revenue in current week'] - dif_byproduct_eachweek ['Revenue in previous week']

dif_byproduct_eachweek .head(5)

,product_name,Revenue in current week,Revenue in previous week,difference
0,Wireless Earbuds,140464,94430,46034
1,Phone Case,51182,44002,7180
2,Screen Protector,23444,18223,5221
3,USB-C Charging Cable,46042,31076,14966
4,Portable Power Bank,88354,64798,23556


- We can use the above table in our model to also tell the owner about the products whose sales have declined or warn about sales of any product declined below 10%.

In [45]:
##Function to calculate the return rate of products in each week
def get_return_rate(df, rate_col_name):
    # Counts total rows (units sold) and sums True values (units returned)
    calc = df.groupby('product_name')['is_return'].agg(sold='count', returned='sum')
    calc[rate_col_name] = 100 * (calc['returned'] / calc['sold'])
    return calc[[rate_col_name]]



# Calculate for both weeks and merge into one DataFrame
return_rate_product_weekly = get_return_rate(last_week, 'Return Rate Last Week').merge(
    get_return_rate(current_week, 'Return Rate Current Week'), 
    on='product_name', 
    how='outer'
).fillna(0).reset_index()


return_rate_product_weekly.head(5)


,product_name,Return Rate Last Week,Return Rate Current Week
0,Laptop Sleeve,9.090909,6.666667
1,Phone Case,25.000000,29.090909
2,Portable Power Bank,7.692308,0.000000
3,Screen Protector,21.875000,8.333333
4,USB-C Charging Cable,22.580645,10.869565


In [46]:
##Return rate grouped by city for each week
#function to compare return rate from each city during each week
def get_return_rate_by_city(df, rate_col_name):
    calc = df.groupby('customer_city')['is_return'].agg(sold='count', returned='sum')
    calc[rate_col_name] = 100 * (calc['returned'] / calc['sold'])
    return calc[[rate_col_name]]

return_rate_bycity_weekly=get_return_rate_by_city(last_week,'Return Rate Last week').merge(
    get_return_rate_by_city(current_week,'Return Rate Last Week'),
    on='customer_city',
    how='outer'
).fillna(0).reset_index()

return_rate_bycity_weekly.head(10)

,customer_city,Return Rate Last week,Return Rate Last Week
0,Faisalabad,11.111111,28.571429
1,Islamabad,14.814815,14.285714
2,Karachi,24.000000,11.111111
3,Lahore,16.949153,10.937500
4,Multan,16.666667,11.111111
5,Peshawar,25.000000,33.333333
6,Quetta,0.000000,0.000000
7,Rawalpindi,33.333333,0.000000


What can be viewed and questioned from this data
- 1.Quetta saw a drop of 50 percent rate this week.Its a good news but what if we have zero orders from quetta this week.
- 2.Return rate overall this week have increased for each city.


What we can do for this:
- 1.Analyze most frequent issue in each city or any new issue reported from any city and suggest a suitable solution to the business man 
- 2.We can also go for check that how many order of COD and how many order of online payment are returned in each cisty to suggest a solution.


In [47]:
##This will show a table in which cities are arranged accoring to sales and the return rate of each payment methos is shown
city_sales_count = current_week['customer_city'].value_counts().sort_values(ascending=True)

payment_return_pct = (current_week.groupby(['customer_city', 'payment_method'])['is_return']
                      .mean()
                      .unstack(fill_value=0) * 100)

# Step 3: Reorder the percentage table rows to match the sorted cities
final_table = payment_return_pct.loc[city_sales_count.index]

# Step 4 (Optional but helpful): Add the total sales count as the first column so you can see the sort
final_table.insert(0, 'Total Orders', city_sales_count)
final_table.head(11)

payment_method,Total Orders,Cash on Delivery,Easypaisa,JazzCash,Online Transfer
customer_city,,,,,
Quetta,1,0.000000,0.000000,0.0,0.000000
Peshawar,3,0.000000,100.000000,0.0,0.000000
Rawalpindi,6,0.000000,0.000000,0.0,0.000000
Multan,9,14.285714,0.000000,0.0,0.000000
Faisalabad,14,40.000000,0.000000,0.0,0.000000
Islamabad,42,15.151515,0.000000,25.0,0.000000
Karachi,54,13.333333,0.000000,0.0,0.000000
Lahore,64,6.818182,33.333333,0.0,27.272727


In [48]:
def analyze_city_returns(df):
    # Ask the user for input
    city_name = input("Enter a city name to analyze (e.g., Karachi, Lahore): ")
    
    # Filter for the specific city (ignoring case) AND only returned items
    city_returns = df[
        (df['customer_city'].str.lower() == city_name.lower()) & 
        (df['is_return'] == True)
    ]
    
    # Handle the case where the city isn't found or has no returns
    if city_returns.empty:
        print(f"No return data found for {city_name}.")
        return
        
    # Count how many times each reason appears and create a clean table
    reason_counts = city_returns['return_reason'].value_counts().reset_index()
    reason_counts.columns = ['Reason for Return', 'Frequency']
    
    # Display the results
    print(f"\n--- Return Analysis for {city_name.title()} ---")
    print(reason_counts)

# To run the program, just call the function and pass your dataframe:
analyze_city_returns(current_week)


--- Return Analysis for Karachi ---
            Reason for Return  Frequency
0                Changed mind          2
1  Different from description          1
2         Sound quality issue          1
3          Poor build quality          1
4          Wrong model / size          1


- The short code is very useful to get to the main issue of why there are return rates in each city

**Phase 3:To go in depth about customers habits**

In [49]:
##Finding the return rates of repeat customer this week
customer_analysis = current_week.groupby('customer_id').agg(
    total_orders=('customer_id', 'count'),  # Counts the number of times the ID appears (total rows)
    return_rate=('is_return', 'mean') # Calculates the return rate
).reset_index()
customer_analysis['customer_type'] = customer_analysis['total_orders'].apply(
    lambda x: 'Repeat Buyer' if x > 1 else 'One-Time Buyer'
)

customer_analysis_onetime=customer_analysis[customer_analysis['customer_type']=="One-Time Buyer"]
customer_analysis_repeat=customer_analysis[customer_analysis['customer_type']=="Repeat Buyer"]

customer_analysis_repeat.head(10)


,customer_id,total_orders,return_rate,customer_type
0,CUST-001,3,0.333333,Repeat Buyer
1,CUST-002,9,0.111111,Repeat Buyer
2,CUST-003,7,0.428571,Repeat Buyer
3,CUST-004,10,0.200000,Repeat Buyer
4,CUST-005,7,0.428571,Repeat Buyer
5,CUST-006,8,0.250000,Repeat Buyer
6,CUST-007,10,0.100000,Repeat Buyer
7,CUST-008,12,0.000000,Repeat Buyer
8,CUST-009,5,0.000000,Repeat Buyer
9,CUST-010,11,0.090909,Repeat Buyer


- Above data is to show that customers returning the product were one time buyers or frequent buyer.We can infer whther the marketing expenditure of attracting these new cutomers is giving any return or not.Moreover we can also guess whats wrong in our website due to which customers are ordering wrong 

**Phase 4:Analyzing the courier services**

In [50]:
courier_return_rates = (current_week.groupby(['customer_city', 'courier'])['is_return']
                        .mean()
                        .unstack(fill_value=0) * 100)

# Round the percentages to 1 decimal place for readability
courier_return_rates = courier_return_rates.round(1)

print(courier_return_rates)

courier        Leopards  Rider  Swyft   TCS
customer_city                              
Faisalabad         37.5    0.0    0.0  16.7
Islamabad           7.7   26.7    0.0   7.1
Karachi             7.7    0.0   19.0   5.0
Lahore              4.2   13.6    0.0  16.7
Multan             12.5    0.0    0.0   0.0
Peshawar            0.0    0.0    0.0  50.0
Quetta              0.0    0.0    0.0   0.0
Rawalpindi          0.0    0.0    0.0   0.0


- This helps to decide which courier is performing good in a specific city and then return rates of a courrier in particular city become much higher along with complains of arrival of faulty products so we can alert them accordingly.

Phase 5:Analyzing the inventory health

In [51]:
import pandas as pd
import numpy as np

# 1. Calculate total sold this week per product
sales_totals = current_week.groupby('product_name')['quantity'].sum().reset_index(name='net_units_sold_this_week')

# 2. Merge your inventory dataframe with the sales totals
report = initial_stock.merge(sales_totals, on='product_name', how='left').fillna(0)

# 3. CALCULATE the remaining stock yourself (Initial Stock minus Units Sold)
report['remaining_stock'] = report['initial_stock'] - report['net_units_sold_this_week']

# 4. Apply the formula safely using np.where to avoid division by zero
report['days_remaining'] = np.where(
    report['net_units_sold_this_week'] > 0,
    report['remaining_stock'] / (report['net_units_sold_this_week'] / 7),
    np.inf
)

# 5. Clean up and display
report['days_remaining'] = report['days_remaining'].round(1)
final_table = report[['product_name', 'net_units_sold_this_week', 'remaining_stock', 'days_remaining']]

final_table.head(5)

,product_name,net_units_sold_this_week,remaining_stock,days_remaining
0,Wireless Earbuds,31,49,11.1
1,Phone Case,83,167,14.1
2,Screen Protector,51,249,34.2
3,USB-C Charging Cable,65,135,14.5
4,Portable Power Bank,27,28,7.3


In [52]:
not_return=current_week.loc[current_week['is_return']==False]
total=(not_return['unit_price_pkr']*not_return['quantity']).sum()
print("The total is=",total)

The total is= 361380
